# Routing workflow with Pydantic AI

Routing classifies the input and sends it to the right agent for specialized handling. The classification can be done by an LLM or a traditional model.

```mermaid
flowchart LR 
    In([In]) --> Router["LLM Call Router"]

    Router -->|Route 1| LLM1["LLM Call 1"]
    Router -->|Route 2| LLM2["LLM Call 2"]
    Router -->|Route 3| LLM3["LLM Call 3"]

    LLM1 --> Out([Out])
    LLM2 --> Out
    LLM3 --> Out
```

**Examples:**
- Classify complexity of question and adjust model depending on it
- Classify type of query and use specialized tools (e.g., indexes, prompts)

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
import logfire
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent

load_dotenv()

logfire.configure()
logfire.instrument_pydantic_ai()

## Vanilla workflow

We use a router agent with structured output to classify the message, then dispatch to the appropriate specialized agent.

In [ ]:
class RouterOutput(BaseModel):
    category: Literal["write_article", "generate_table_of_contents", "review_article"]


router_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are a helpful assistant. You will classify the message into one of the following categories: "
        "'write_article', 'generate_table_of_contents', 'review_article'."
    ),
    output_type=RouterOutput,
)

agent_writer = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a writer. You will write an article about the topic provided.",
)

agent_toc = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are an expert writer specialized in SEO. Provided with a topic, "
        "you will generate the table of contents for a short article."
    ),
)

agent_reviewer = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a writer. You will review the article for the topic provided.",
)


def run_workflow(message: str) -> str:
    router_output = router_agent.run_sync(f"Classify the message: {message}")
    category = router_output.output.category
    if category == "write_article":
        return agent_writer.run_sync(f"Write an article about {message}").output
    elif category == "generate_table_of_contents":
        return agent_toc.run_sync(
            f"Generate the table of contents of an article about {message}"
        ).output
    else:
        return agent_reviewer.run_sync(
            f"Review the article for the topic {message}"
        ).output

In [ ]:
toc = run_workflow("Generate a table of contents for an article about AI")
print(toc)

In [ ]:
review = run_workflow(
    "Review this post: 'There are times where there's no time, so you don't have time to write an article about it.'"
)
print(review)

## Exercise

Implement a routing workflow that takes the content of a PDF file and depending on the type of document, processes it in a different way. Mock the processing for now.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader


class DocumentRoute(BaseModel):
    category: Literal["financial", "legal", "marketing", "pets", "other"]


pdf_router = Agent(
    "openai:gpt-5-mini",
    output_type=DocumentRoute,
    system_prompt=(
        "Classify the document into one of these categories: "
        "'financial', 'legal', 'marketing', 'pets', or 'other'."
    ),
)


def get_first_n_pages(file_path: str, n: int = 5) -> str:
    loader = PyPDFLoader(file_path)
    pages = []
    for page in loader.lazy_load():
        pages.append(page.page_content)
        if len(pages) == n:
            break
    return "\n\n".join(pages)


def process_financial_document(document: str) -> str:
    return "Financial document processed: extracted account limits and banking details."


def process_legal_document(document: str) -> str:
    return "Legal document processed: extracted obligations, clauses, and parties."


def process_marketing_document(document: str) -> str:
    return "Marketing document processed: extracted campaign goals and messaging."


def process_pets_document(document: str) -> str:
    return "Pets document processed: extracted breed, care, and behavior details."


def process_other_document(document: str) -> str:
    return "Other document processed: extracted a generic summary."


processors = {
    "financial": process_financial_document,
    "legal": process_legal_document,
    "marketing": process_marketing_document,
    "pets": process_pets_document,
    "other": process_other_document,
}


def run_pdf_workflow(path: str) -> dict:
    document = get_first_n_pages(path)
    route = pdf_router.run_sync(
        f"Classify this document:\n\n{document}"
    ).output.category
    output = processors[route](document)
    return {"category": route, "output": output}

In [ ]:
print(run_pdf_workflow("assets/bbva.pdf"))
print(run_pdf_workflow("assets/dogs.pdf"))